# ①' 基礎編【解答版】：Python・データ分析・可視化の第一歩
**岡山県 高校教員向け データサイエンス・ハンズオン（1時間目）**

このノートは**写経（コードをそのまま打って実行）**しながら進めます。
Chromebook + Google Colab だけで、インストール不要・無料で動きます。

学ぶこと：
1. Colab と Python の基本
2. `pandas` で本物の公開データ（国勢調査）を読む
3. 集計する・並べ替える
4. `matplotlib` でグラフにする

> **Colabの使い方**：セルを選んで **Shift+Enter** で実行。左上「＋コード」で新しいセルを追加。


## 0. まずは準備（下の2つのセルを実行）

In [ ]:
# === データ読み込みの準備（このセルを最初に実行）===
import pandas as pd, io, os

# データの入手方法：
#  ・手動アップロード（既定）：BASE_URL は "" のまま。実行すると選択画面が出るのでCSVを選ぶ。
#  ・公開リポジトリを使う場合のみ：BASE_URL に raw URL を入れる
#    例) "https://raw.githubusercontent.com/USER/REPO/main/data/"
#    ※ Privateリポジトリの raw URL はトークンが必要なため受講者配布には不向き。手動アップロードを推奨。
BASE_URL = "https://raw.githubusercontent.com/yasuyuki-nogami/ds-handson/main/data/"

def load(name):
    """data/名前.csv を読み込む。ローカル→(URL)→手動アップロードの順に試す。"""
    if os.path.exists("data/" + name):
        return pd.read_csv("data/" + name)
    if BASE_URL:
        try:
            return pd.read_csv(BASE_URL + name)
        except Exception:
            pass
    from google.colab import files
    print(name, " をアップロードしてください")
    up = files.upload()
    key = list(up.keys())[0]
    return pd.read_csv(io.BytesIO(up[key]))

print("準備OK：load('ファイル名.csv') でデータを読み込めます")


In [ ]:
# 日本語をグラフに表示できるようにする（Colabでは最初に1回だけ）
!pip -q install japanize-matplotlib
import matplotlib.pyplot as plt
import japanize_matplotlib   # これでグラフの日本語が文字化けしない
print("日本語フォント準備OK")


## 1. Python 超入門
プログラミングは「**箱（変数）に値を入れて、計算し、表示する**」の繰り返しです。

In [ ]:
# 変数：名前をつけて値を保存する
name = "岡山"
year = 2020
pi = 3.14
print(name, year, pi)


In [ ]:
# 計算してみる
a = 10
b = 3
print("足し算", a + b)
print("割り算", a / b)
print("あまり", a % b)


In [ ]:
# リスト：値をまとめて入れる箱
scores = [55, 70, 65, 90, 40]
print("個数", len(scores))
print("合計", sum(scores))
print("平均", sum(scores) / len(scores))


In [ ]:
# for：繰り返し
for s in scores:
    print("点数は", s)


In [ ]:
# if：条件分岐
for s in scores:
    if s >= 60:
        print(s, "合格")
    else:
        print(s, "不合格")


In [ ]:
# 関数：処理に名前をつけて再利用する
def hantei(x):
    if x >= 60:
        return "合格"
    return "不合格"

print(hantei(80))
print(hantei(50))


## 2. 本物の公開データを読む（pandas）
`pandas` は表データ（Excelのような表）を扱う道具です。
ここで使う **`prefecture.csv` は、総務省「国勢調査(2020年)」等にもとづく47都道府県の人口・面積の実データ**です。

In [ ]:
df = load("prefecture.csv")
df.head()   # 先頭5行を表示


In [ ]:
print("行数・列数：", df.shape)   # (行, 列)
print("列の名前：", list(df.columns))


In [ ]:
df.describe()   # 数値列の要約統計量（平均・最大・最小など）


## 3. 選ぶ・絞る・並べ替える
表から必要な部分だけを取り出します。

In [ ]:
# 列を取り出す
df["population_2020"].head()


In [ ]:
# 条件で行を絞る：人口が300万人以上の都道府県
df[df["population_2020"] >= 3_000_000]


In [ ]:
# 並べ替え：人口が多い順トップ10
top10 = df.sort_values("population_2020", ascending=False).head(10)
top10[["prefecture", "population_2020"]]


In [ ]:
# 岡山県のデータだけ見る
df[df["prefecture"] == "岡山県"]


## 4. 集計する（groupby）
地方（region）ごとに平均を出してみます。

In [ ]:
df.groupby("region")["population_2020"].mean().sort_values(ascending=False)


## 5. グラフにする（matplotlib）
数字の表を「絵」にすると特徴が一目で分かります。ここでは3種類を描きます。
- 横棒グラフ（ランキングを見やすく）
- 地方別の集計を棒グラフに
- 散布図で「2つの数に関係があるか」を確かめる（相関係数つき）

In [ ]:
# ① 横棒グラフ：人口トップ10（数が多い順に見やすく）
top10 = df.sort_values("population_2020", ascending=False).head(10)
top10 = top10.sort_values("population_2020")   # 横棒は下から大きくなるよう並べ替え
plt.figure(figsize=(7,4))
plt.barh(top10["prefecture"], top10["population_2020"] / 10000)
plt.title("人口が多い都道府県トップ10（2020年）")
plt.xlabel("人口（万人）")
plt.tight_layout()
plt.show()


In [ ]:
# ② 地方別の人口合計（4.のgroupbyの結果をグラフにする）
region_pop = df.groupby("region")["population_2020"].sum().sort_values()
plt.figure(figsize=(7,4))
plt.barh(region_pop.index, region_pop.values / 10000, color="darkorange")
plt.title("地方別の人口合計")
plt.xlabel("人口（万人）")
plt.tight_layout()
plt.show()


In [ ]:
# ③ 散布図：人口と人口密度に関係はある？（点の並びで確かめる）
plt.figure(figsize=(6,5))
plt.scatter(df["population_2020"] / 10000, df["density_per_km2"])
plt.title("人口 vs 人口密度")
plt.xlabel("人口（万人）")
plt.ylabel("人口密度（人/km²）")
plt.show()

# 相関係数（1に近いほど「一方が大きいと他方も大きい」）
r = df["population_2020"].corr(df["density_per_km2"])
print(f"相関係数 = {r:.2f}  → 人口が多い都道府県ほど人口密度も高い傾向")


## 6. 練習（やってみよう）
1. 人口密度が高い順トップ5を表示してみよう（`sort_values`）。
2. 面積が最も大きい都道府県はどこ？
3. 地方（region）ごとの**面積の合計**を出してみよう（`groupby` と `sum`）。

> ヒント：これまでのセルをコピーして少し書き換えるだけでできます。

In [ ]:
# 【解答例】
# 1. 人口密度が高い順トップ5
print("■ 人口密度トップ5")
print(df.sort_values("density_per_km2", ascending=False).head(5)[["prefecture","density_per_km2"]])

# 2. 面積が最も大きい都道府県
print("\n■ 面積が最大の都道府県")
print(df.sort_values("area_km2", ascending=False).head(1)[["prefecture","area_km2"]])

# 3. 地方ごとの面積の合計
print("\n■ 地方別の面積合計")
print(df.groupby("region")["area_km2"].sum().sort_values(ascending=False))


### まとめ
- `pandas` で公開データを読み、`describe / sort_values / groupby` で集計できた
- `matplotlib` で棒・ヒスト・散布図が描けた
- **これらは全部、無料の公開データ + Colab だけでできる！**

次（2時間目）は、自分で手を動かす**応用ハンズオン**に進みます。